# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage tracks two distinct wallet groups on the full trade stream (BUY + SELL):

- `copyable_group`: wallets that pass base eligibility and strong copyability thresholds.
- `predicting_group`: the remaining eligible wallets (not in copyable) that pass a predictability score threshold.

Notes:
- Group membership is threshold-based (absolute scores), not target-count based.
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- `selected_wallets_v2.parquet` stores both groups with a `wallet_group` label.
- Stage2 backtest uses only `predicting_group`.


In [61]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [62]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [63]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-06-01  TRAIN_A_END_DATE=2026-04-01


## Compute / load wallet skill metrics (legacy diagnostics)

These open-buy metrics are not used for the final copy-trigger wallet selection.


In [64]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [65]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [66]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (full-stream copy-trigger path)


## Build wallet profiles and signal events

In [67]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [68]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [69]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [70]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)

df_slice = df_full[df_full["is_train"]].copy()
wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))


42949


In [71]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(
    wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)


In [72]:
wallet_cohorts = {}

In [73]:
print(len(wallet_vol_train))
wallet_vol_train.head()


42949


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0xa231231e1e0a4d69fd7e51c3c6e6676b3f692e37,13.70,14,4,17,514.16,124.69,321.45,1.03,1.00,0.49,1.04,0.96,0.92,0.50,165.67,2025-12-22 15:47:30+00:00,118.12,307.14,2.46,0.20,0.00,0.24,1.00,118.12
1,0x3c6385912d528e4a71a5fb47ef269c960ec7a0df,5.17,2,1,20,40.14,376.45,391.36,1.00,1.00,1.00,1.00,1.00,1.00,0.50,99.00,2026-05-24 11:30:00+00:00,99.00,38.05,0.10,23.15,0.06,9.38,1.00,99.00
2,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1,1.52,150.48,150.48,1.00,1.00,1.00,1.00,1.00,1.00,1.00,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
3,0x8da0d29ec66508950ad72846ac3164db7cc41c53,NaN,1,1,1,2.50,205.83,205.83,1.00,1.00,1.00,1.00,1.00,1.00,1.00,82.33,2026-04-03 19:40:00+00:00,82.33,0.00,0.00,0.00,0.00,82.33,1.00,82.33
4,0xdff348c95aa09169740de5a2d4ac8019bb3f531d,3.74,2,1,2,101.57,94.50,94.50,1.00,1.00,1.00,1.00,1.00,1.00,0.50,61.50,2026-05-14 22:35:00+00:00,61.50,100.00,1.06,100.00,1.06,0.93,1.00,61.50


In [74]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['0x0a'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [75]:
# Final wallet selection: threshold-based copyable + predicting groups
# COPYABLE_SCORE_MIN = -1
# PREDICTING_SCORE_MIN = -2

candidates = wallet_vol_train.copy()

for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

base_mask = (
    (candidates['average_roi'] >= 0.1)
    # & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.2)
    & (candidates['top_market_pnl_pct'] < 0.25)
    & (candidates['top5_pnl_pct'] < 0.55)
    & (candidates['top10_pnl_pct'].fillna(0.85) < 0.8)
    & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.25)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (END_DATE_TRAIN - pd.Timedelta(days=60)))
    & (candidates['total_notional'] >= 5_000)
)

eligible_base = candidates[base_mask].copy()
if eligible_base.empty:
    raise ValueError('No wallets passed base eligibility filters.')

eligible_base['sample_mult'] = np.clip(
    np.log1p(eligible_base['num_buckets']) / np.log(2000.0),
    0.20,
    1.00,
)
eligible_base['downside_tail'] = eligible_base['worst5_pnl_pct'].abs().fillna(0.0)
eligible_base['predictability_score'] = eligible_base['sample_mult'] * (
    # 2.5 * eligible_base['average_roi'].fillna(0.0)
    # + 1.5 * eligible_base['median_roi'].fillna(0.0)
    + 1.2 * (eligible_base['positive_bucket_share'].fillna(0.5) - 0.5)
    - 1.0 * eligible_base['pnl_volatility'].fillna(0.0)
    - 1.2 * eligible_base['max_drawdown_to_pnl'].fillna(0.0)
    - 0.8 * eligible_base['top_market_abs_pnl_pct'].fillna(eligible_base['top_market_pnl_pct']).fillna(0.0)
    - 0.6 * eligible_base['market_pnl_hhi'].fillna(0.0)
    - 0.5 * eligible_base['downside_tail']
)
eligible_base['predicting_final_score'] = eligible_base['predictability_score']

copyable_mask = (
    (eligible_base['copyable_pnl'] > 0)
    & (eligible_base['average_roi'] >= 0.04)
    # & (eligible_base['copyable_pnl_factor'] >= 0.1)
    # & (eligible_base['copyable_roi'] >= 0.01)
    # & (eligible_base['median_roi'] >= 0.00)
    # & (eligible_base['average_roi'] >= 0.04)
    # & (
    #     eligible_base['max_copyable_drawdown_to_copyable_pnl']
    #     .fillna(eligible_base['max_drawdown_to_pnl'])
    #     .fillna(0.0)
    #     <= 0.3
    # )
)
copyable_candidates = eligible_base[copyable_mask].copy()
copyable_candidates['copyable_efficiency'] = copyable_candidates['copyable_pnl'].fillna(0.0) / (copyable_candidates['total_notional'].fillna(0.0) + 1.0)
copyable_candidates['copyable_dd_ratio'] = (
    copyable_candidates['max_copyable_drawdown_to_copyable_pnl']
    .fillna(copyable_candidates['max_drawdown_to_pnl'])
    .fillna(0.0)
)
copyable_candidates['copyable_score'] = (
    1.8 * copyable_candidates['copyable_roi'].fillna(0.0)
    + 1.2 * copyable_candidates['copyable_pnl_factor'].fillna(0.0)
    + 25.0 * copyable_candidates['copyable_efficiency'].clip(lower=-1.0, upper=1.0)
    - 0.8 * copyable_candidates['copyable_dd_ratio'].clip(lower=0.0)
    - 0.5 * copyable_candidates['top_market_abs_pnl_pct'].fillna(copyable_candidates['top_market_pnl_pct']).fillna(0.0)
)
copyable_candidates['final_score'] = (
    0.60 * copyable_candidates['predictability_score']
    + 0.40 * copyable_candidates['copyable_score']
)

wallet_cohorts['copyable_group'] = (
    copyable_candidates
    .sort_values('final_score', ascending=False)
    .reset_index(drop=True)
)

predicting_pool = eligible_base[
    ~eligible_base['wallet'].isin(wallet_cohorts['copyable_group']['wallet'])
].copy()
wallet_cohorts['predicting_group'] = (
    predicting_pool
    # predicting_pool[predicting_pool['predicting_final_score'] >= PREDICTING_SCORE_MIN]
    .sort_values('predicting_final_score', ascending=False)
    .reset_index(drop=True)
)

if wallet_cohorts['copyable_group'].empty:
    raise ValueError('No wallets passed copyable-group thresholds.')
if wallet_cohorts['predicting_group'].empty:
    raise ValueError('No wallets passed predicting-group threshold.')

overlap = set(wallet_cohorts['copyable_group']['wallet']) & set(wallet_cohorts['predicting_group']['wallet'])
if overlap:
    raise ValueError(f'Wallet groups must be disjoint; found overlap of {len(overlap)} wallets.')

wallet_cohorts['copyable_group']['wallet_quality'] = wallet_cohorts['copyable_group']['final_score']
wallet_cohorts['predicting_group']['wallet_quality'] = wallet_cohorts['predicting_group']['predicting_final_score']

wallet_cohorts['multi_filter'] = wallet_cohorts['copyable_group'].copy()

print(f"Base-eligible wallets: {len(eligible_base):,}")
print(f"Copyable candidates: {len(copyable_candidates):,}")
print(f"Selected wallets (copyable group): {len(wallet_cohorts['copyable_group']):,}")
print(f"Selected wallets (predicting group): {len(wallet_cohorts['predicting_group']):,}")
wallet_cohorts['copyable_group'].head(10)



Base-eligible wallets: 106
Copyable candidates: 71
Selected wallets (copyable group): 71
Selected wallets (predicting group): 35


,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
0,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.83,69,38,80,8009.87,1145.21,687.19,0.49,0.71,-0.24,0.13,0.09,0.05,0.73,0.18,2026-02-28 16:20:00+00:00,0.73,100.00,0.09,73.91,0.11,0.14,0.60,0.44,0.56,0.24,-0.50,-0.50,0.09,0.11,3.52,1.11,1.11
1,0x553a95b3c1b474d6c4b2b48772a8152c25f3177f,1.04,265,46,356,64178.58,2181.59,1028.09,0.40,0.61,-0.08,0.19,0.17,0.10,0.68,0.02,2026-03-05 02:30:00+00:00,0.71,396.00,0.18,173.72,0.17,0.03,0.47,0.33,0.73,0.08,-0.93,-0.93,0.02,0.17,1.34,-0.02,-0.02
2,0xb9c1ea9b9ae211e384e7128255e8387edb380791,1.37,652,263,753,20706.26,4456.92,1100.73,0.33,0.46,-0.12,0.15,0.08,0.02,0.70,0.29,2026-01-27 16:47:30+00:00,0.36,441.23,0.10,201.37,0.18,0.22,0.25,0.09,0.85,0.12,-1.17,-1.17,0.05,0.18,1.60,-0.06,-0.06
3,0x99984e22205053950eb25453779267bcc1aee858,0.34,3486,1075,4147,34309.07,3724.28,441.06,0.20,0.32,-0.13,0.05,0.03,0.00,0.50,-1.00,2025-09-19 14:35:00+00:00,1.39,142.87,0.04,123.69,0.28,0.11,0.12,0.16,1.00,0.13,-0.47,-0.47,0.01,0.28,0.52,-0.07,-0.07
4,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.36,221,47,262,50819.14,6107.90,1358.21,0.40,0.62,-0.21,0.14,0.12,0.07,0.77,0.12,2025-02-23 16:25:00+00:00,1.07,631.30,0.10,449.21,0.33,0.12,0.22,0.24,0.71,0.21,-1.00,-1.00,0.03,0.33,1.04,-0.18,-0.18
5,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.78,12293,2823,16091,965438.97,49898.42,16492.87,0.22,0.36,-0.15,0.15,0.04,0.01,0.67,0.07,2025-10-28 03:50:00+00:00,0.11,4668.08,0.09,3498.04,0.21,0.05,0.33,0.04,1.00,0.15,-0.80,-0.80,0.02,0.21,0.70,-0.20,-0.20
6,0x269af8d4c4a275572ae030d5da54c7831efa190e,0.53,423,152,536,13802.42,754.20,257.21,0.36,0.57,-0.23,0.19,0.14,0.04,0.61,0.01,2026-03-14 01:25:00+00:00,0.34,127.00,0.17,189.89,0.74,0.05,0.34,0.11,0.80,0.23,-0.68,-0.68,0.02,0.74,0.42,-0.24,-0.24
7,0xca12d28728c46d3484395243f5f842f2c321ea03,0.74,1462,497,2066,265362.62,5092.72,1306.08,0.45,0.58,-0.17,0.14,0.11,0.04,0.79,0.01,2026-03-08 07:40:00+00:00,0.18,406.82,0.08,135.99,0.10,0.02,0.26,0.05,0.96,0.17,-0.66,-0.66,0.00,0.10,0.38,-0.25,-0.25
8,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.92,3812,1106,4714,225426.08,29006.48,4236.02,0.20,0.32,-0.05,0.11,0.06,0.01,0.61,0.02,2025-10-13 11:25:00+00:00,0.45,745.47,0.03,555.95,0.13,0.13,0.15,0.07,1.00,0.05,-0.90,-0.90,0.02,0.13,0.63,-0.29,-0.29
9,0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,0.48,2013,721,2378,50708.41,2647.25,297.75,0.22,0.37,-0.22,0.06,0.05,0.01,0.67,0.05,2026-01-14 13:15:00+00:00,0.60,392.78,0.15,73.49,0.25,0.05,0.11,0.07,1.00,0.22,-0.61,-0.61,0.01,0.25,0.18,-0.29,-0.29


In [76]:
copyable_candidates.sort_values('trade_count', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score
15577,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0.13,15858,1780,23101,169288.97,2789.95,4.80,0.22,0.32,-0.09,0.05,0.04,0.01,0.45,-1.00,2026-01-22 00:35:00+00:00,0.11,71.70,0.03,258.79,53.87,0.02,0.00,0.00,1.00,0.09,-0.31,-0.31,0.00,53.87,-43.11,-17.43
10545,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,2.10,15434,1364,22905,1979741.13,66432.79,19834.33,0.34,0.50,-0.30,0.09,0.03,0.01,0.50,-0.31,2026-01-07 07:00:00+00:00,0.16,6745.19,0.10,7397.36,0.37,0.03,0.30,0.05,1.00,0.30,-2.40,-2.40,0.01,0.37,0.38,-1.29
11175,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.78,12293,2823,16091,965438.97,49898.42,16492.87,0.22,0.36,-0.15,0.15,0.04,0.01,0.67,0.07,2025-10-28 03:50:00+00:00,0.11,4668.08,0.09,3498.04,0.21,0.05,0.33,0.04,1.00,0.15,-0.80,-0.80,0.02,0.21,0.70,-0.20
9262,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,2.44,6252,1170,11032,3501602.26,95291.81,23792.36,0.33,0.49,-0.21,0.10,0.06,0.01,0.56,0.01,2025-10-10 17:00:00+00:00,0.33,10193.16,0.11,6836.91,0.29,0.03,0.25,0.08,1.00,0.21,-2.65,-2.65,0.01,0.29,0.36,-1.45
8491,0x2a66d9fd6fa7f9244cef81c208046cb9ee33a177,0.38,6431,857,9951,83172.29,1028.04,109.92,0.54,0.80,-0.40,0.12,0.08,0.02,0.46,-1.00,2026-03-31 01:45:00+00:00,1.00,153.13,0.15,378.96,3.45,0.01,0.11,0.11,1.00,0.40,-0.88,-0.88,0.00,3.45,-2.44,-1.51
10684,0x0bfb8009df6c46c1fdd79b65896cf224dc4526a7,1.47,6946,1200,9833,1024059.19,37825.10,7055.20,0.51,0.70,-0.22,0.14,0.05,0.01,0.60,0.02,2026-03-26 13:55:00+00:00,0.25,5952.25,0.16,7722.11,1.09,0.04,0.19,0.05,1.00,0.22,-1.69,-1.69,0.01,1.09,-0.42,-1.19
12221,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,1.20,6678,1934,9223,453495.48,68957.04,7867.79,0.13,0.20,-0.10,0.03,0.02,0.01,0.64,0.03,2026-03-09 22:55:00+00:00,0.19,2284.04,0.03,3811.31,0.48,0.15,0.11,0.02,1.00,0.10,-1.14,-1.14,0.02,0.48,0.21,-0.60
8101,0x41583f2efc720b8e2682750fffb67f2806fece9f,2.28,6134,1073,8860,1059343.30,79599.91,25513.18,0.23,0.39,-0.22,0.06,0.03,0.00,0.48,-1.00,2025-10-30 18:35:00+00:00,0.38,6430.97,0.08,9436.71,0.37,0.08,0.32,0.12,1.00,0.22,-2.54,-2.54,0.02,0.37,0.90,-1.16
9647,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0.95,6346,2015,8201,394381.22,50990.26,4398.12,0.13,0.20,-0.06,0.06,0.03,0.00,0.51,0.00,2025-12-08 20:50:00+00:00,0.81,1863.33,0.04,2199.27,0.50,0.13,0.09,0.07,1.00,0.06,-1.04,-1.04,0.01,0.50,0.10,-0.58
10186,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.79,6470,2053,8014,649987.91,94822.91,16316.84,0.35,0.46,-0.10,0.20,0.06,0.01,0.56,0.07,2026-01-27 03:40:00+00:00,0.33,8553.27,0.09,4556.81,0.28,0.15,0.17,0.06,1.00,0.10,-1.93,-1.93,0.03,0.28,0.68,-0.88


In [77]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


In [78]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['dt'] > pd.to_datetime("2026-04-16", utc=True))
    # & (df_full['wallet'] == '0xbacd00c9080a82ded56f504ee8810af732b0ab35')
    & (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(20)

278262.626761
169405.859841


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
7206,"US x Iran permanent peace deal by June 15, 2026?",0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,No,-703.39,159,10,34391.80,0.02
7207,"US x Iran permanent peace deal by June 15, 2026?",0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,Yes,33688.40,465,14,34391.80,0.02
3337,"Will Trump say ""Iran"" during events with Xi Jinping?",0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,No,16317.24,306,14,17157.03,0.05
3338,"Will Trump say ""Iran"" during events with Xi Jinping?",0xaa145cf5714455f546afe53037b8712df749ba96c04820c2d41f37f877f29697,Yes,839.80,60,6,17157.03,0.05
6041,Will Donald Trump announce that the United States blockade of the Strait of Hormuz has been lift...,0x4d0c4865bdecc5f7971dbab47bb6c069d93909dec75bdff52fb766ae6094a6d6,No,-265.43,97,4,8966.69,0.03
6040,Will Donald Trump announce that the United States blockade of the Strait of Hormuz has been lift...,0x4d0c4865bdecc5f7971dbab47bb6c069d93909dec75bdff52fb766ae6094a6d6,Yes,8701.26,149,8,8966.69,0.03
7842,Iran closes its airspace by May 27?,0xdb22a7749b831aa07a52cbc83213e6c8ceb88226b224a831512f4460011bb0a1,Yes,16.08,9,3,7980.35,0.00
7841,Iran closes its airspace by May 27?,0xdb22a7749b831aa07a52cbc83213e6c8ceb88226b224a831512f4460011bb0a1,No,7964.27,79,7,7980.35,0.00
7586,"Russia x Ukraine ceasefire by May 31, 2026?",0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad17755f9cee123c7099b35,Yes,456.12,62,4,7584.82,0.06
7585,"Russia x Ukraine ceasefire by May 31, 2026?",0xa93b28a6384aefff4e7d5adb2061c444e4a0e95b8ad17755f9cee123c7099b35,No,-7128.70,135,10,7584.82,0.06


In [79]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,predicting_final_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
67,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,14.46,4073,620,7538,3898743.59,352501.25,66103.70,0.49,0.77,-0.39,0.20,0.11,0.02,0.53,0.01,2025-11-26 17:50:00+00:00,0.12,53241.93,0.15,50759.68,0.77,0.09,0.19,0.02,1.00,0.39,-14.90,-14.90,0.02,0.77,0.02,-8.93,-8.93
63,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,10.54,1001,294,1886,1187740.15,163745.70,46706.82,0.43,0.66,-0.26,0.09,0.07,0.03,0.61,0.05,2025-12-19 22:20:00+00:00,0.79,13625.20,0.08,10133.74,0.22,0.14,0.29,0.23,0.91,0.26,-9.74,-9.74,0.04,0.22,1.52,-5.24,-5.24
56,0x1c144e30f405a25f991cbd8baa15d40599090869,3.67,4301,1081,5798,626097.31,81785.57,27809.32,0.36,0.54,-0.15,0.10,0.04,0.01,0.61,0.12,2026-01-27 23:00:00+00:00,0.20,10118.78,0.12,4941.62,0.18,0.13,0.34,0.07,1.00,0.15,-3.81,-3.81,0.04,0.18,1.48,-1.69,-1.69
59,0x6df6e2a9ba1e8d7609daada0a83138817f4a8458,3.74,3692,1014,5382,928608.40,85448.24,26016.24,0.31,0.49,-0.23,0.13,0.04,0.01,0.50,-1.00,2025-09-26 02:00:00+00:00,0.39,12404.99,0.15,8691.36,0.33,0.09,0.30,0.12,1.00,0.23,-4.06,-4.06,0.03,0.33,1.00,-2.04,-2.04
42,0x41583f2efc720b8e2682750fffb67f2806fece9f,2.28,6134,1073,8860,1059343.30,79599.91,25513.18,0.23,0.39,-0.22,0.06,0.03,0.00,0.48,-1.00,2025-10-30 18:35:00+00:00,0.38,6430.97,0.08,9436.71,0.37,0.08,0.32,0.12,1.00,0.22,-2.54,-2.54,0.02,0.37,0.90,-1.16,-1.16
52,0x1c1e841584db14084e10e7dca2ad0ab7b60dbfe7,2.44,6252,1170,11032,3501602.26,95291.81,23792.36,0.33,0.49,-0.21,0.10,0.06,0.01,0.56,0.01,2025-10-10 17:00:00+00:00,0.33,10193.16,0.11,6836.91,0.29,0.03,0.25,0.08,1.00,0.21,-2.65,-2.65,0.01,0.29,0.36,-1.45,-1.45
44,0x88b59d79b6e1659c95a0043028e5bb7a26e6205c,4.19,296,36,647,475683.39,59932.81,23172.65,0.44,0.67,-0.15,0.18,0.17,0.08,0.84,0.16,2025-06-23 19:57:30+00:00,0.19,2775.43,0.05,1690.00,0.07,0.13,0.39,0.08,0.75,0.15,-3.07,-3.07,0.05,0.07,1.67,-1.17,-1.17
50,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,2.10,15434,1364,22905,1979741.13,66432.79,19834.33,0.34,0.50,-0.30,0.09,0.03,0.01,0.50,-0.31,2026-01-07 07:00:00+00:00,0.16,6745.19,0.10,7397.36,0.37,0.03,0.30,0.05,1.00,0.30,-2.40,-2.40,0.01,0.37,0.38,-1.29,-1.29
5,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.78,12293,2823,16091,965438.97,49898.42,16492.87,0.22,0.36,-0.15,0.15,0.04,0.01,0.67,0.07,2025-10-28 03:50:00+00:00,0.11,4668.08,0.09,3498.04,0.21,0.05,0.33,0.04,1.00,0.15,-0.80,-0.80,0.02,0.21,0.70,-0.20,-0.20
36,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,1.79,6470,2053,8014,649987.91,94822.91,16316.84,0.35,0.46,-0.10,0.20,0.06,0.01,0.56,0.07,2026-01-27 03:40:00+00:00,0.33,8553.27,0.09,4556.81,0.28,0.15,0.17,0.06,1.00,0.10,-1.93,-1.93,0.03,0.28,0.68,-0.88,-0.88


In [80]:
df_slice.head(1)

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x00000ba68703bce9c2ff4be7177145c1bb3e9ac5,0x098afead2c41677b0f09ae9d0013ca520eacdb3f0d7c8985c6581de7a0e9ac37,2026-05-18 05:53:53+00:00,BUY,5000.00,5000.00,0.14,700.00,0.00,-700.00,-56.59,False,0.00,2026-05-22 06:11:42+00:00,0x0608e4eeb4f28612dfee57a202a790d2f183d4e251f0fcd7dd682685834d9d4d,1,True,404.23,52.55,2.00,2026-05-21T00:00:00Z,Iran closes its airspace by May 21?,"[Politics, Iran, Geopolitics, U.S. x Iran]",Politics,112135611227049632102833296892649508522129989874688677403193353750752980311218,Yes,-700.00,700.00


In [81]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_full[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

9018538
23919


In [82]:
wallet_set = set(wallet_cohorts['copyable_group']['wallet'])
print(f"Copyable wallets: {len(wallet_set)} with fills {df_wallets[df_wallets['wallet'].isin(wallet_set)].shape[0]} with total PnL: {df_wallets[df_wallets['wallet'].isin(wallet_set)]['pnl'].sum():,.2f}")
df_wallets[df_wallets['wallet'].isin(wallet_set)]['trade_pnl'].sum()

Copyable wallets: 71 with fills 23919 with total PnL: 93,463.42


np.float64(93463.42362199999)

In [83]:
df_wallets.head()

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
69595,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x0ee1cc72e03093ab283c450725fc90293d4274fd9adcfff3555e9e137c5373b6,2026-06-04 02:51:19+00:00,BUY,7.72,7.72,0.03,0.23,0.00,-0.23,-0.23,False,0.00,2026-06-04 03:11:54+00:00,0x4cb22409dae4244cccc1648846d476ccb7277ab89bd495197a0d7361a699c6be,1,False,7.72,116.62,2.00,2026-06-20T00:00:00Z,Will Spurs win the 2026 NBA Finals 4-0 be the exact series outcome?,"[Basketball, NBA, NBA Finals, NBA Playoffs, NBA All-Star, 2026 NBA Playoffs]",Basketball,11707185583691257175611487348607070949295883677423660950889660153248542531618,Yes,-0.23,0.23
69596,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x0ee1cc72e03093ab283c450725fc90293d4274fd9adcfff3555e9e137c5373b6,2026-06-04 02:56:28+00:00,SELL,0.00,7.72,0.04,0.31,0.00,0.31,0.00,False,0.00,2026-06-04 03:11:54+00:00,0x0a21d0adfafd6d70f0fa351ec04141d6b4eea074055b8f3a82803152e926b534,1,False,0.00,0.00,0.00,2026-06-20T00:00:00Z,Will Spurs win the 2026 NBA Finals 4-0 be the exact series outcome?,"[Basketball, NBA, NBA Finals, NBA Playoffs, NBA All-Star, 2026 NBA Playoffs]",Basketball,11707185583691257175611487348607070949295883677423660950889660153248542531618,Yes,0.31,7.41
69597,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x0329ad3a1a02a889f24b3d5f4f2e536d1726e4d8fffc992cf1ab7d420c3c3d4c,2026-06-04 03:08:03+00:00,BUY,32.12,5.00,0.00,0.01,0.00,-0.01,0.00,False,0.00,2026-06-04 03:19:00+00:00,0xd97c89608b6c88c7a6be3d6f95166e49a69c3ec5c4004087dd39e0199b8a6826,1,False,0.00,0.00,0.00,2026-06-02T00:00:00Z,Will Dory Benami advance from the CA-32 primary election?,"[Politics, Elections, Primaries, US Election, House Primary, California Primary, June 2 Primaries]",Politics,39164698072766577416104727143028942994587200190194357039087668763260651876310,Yes,-0.01,0.01
69598,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x0329ad3a1a02a889f24b3d5f4f2e536d1726e4d8fffc992cf1ab7d420c3c3d4c,2026-06-04 03:08:03+00:00,BUY,27.12,5.00,0.00,0.01,0.00,-0.01,0.00,False,0.00,2026-06-04 03:19:00+00:00,0x1cacdf4abeffcbe292f1d99c3242d392792ba343e894c6cda7dc70d28825d5a7,1,False,0.00,0.00,0.00,2026-06-02T00:00:00Z,Will Dory Benami advance from the CA-32 primary election?,"[Politics, Elections, Primaries, US Election, House Primary, California Primary, June 2 Primaries]",Politics,39164698072766577416104727143028942994587200190194357039087668763260651876310,Yes,-0.01,0.01
69599,0x0ad7f3411bc87f0b5362155e638f04ef05700971,0x0329ad3a1a02a889f24b3d5f4f2e536d1726e4d8fffc992cf1ab7d420c3c3d4c,2026-06-04 03:08:03+00:00,BUY,22.12,5.00,0.00,0.01,0.00,-0.01,0.00,False,0.00,2026-06-04 03:19:00+00:00,0x8b04bd0e015ace2ac380c7a968c9b431c431376570fe097584435c052b9a55e1,1,False,0.00,0.00,0.00,2026-06-02T00:00:00Z,Will Dory Benami advance from the CA-32 primary election?,"[Politics, Elections, Primaries, US Election, House Primary, California Primary, June 2 Primaries]",Politics,39164698072766577416104727143028942994587200190194357039087668763260651876310,Yes,-0.01,0.01


In [84]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id                   0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
end_date_iso                                                                 2026-01-31T00:00:00Z
question                                                    US strikes Iran by February 28, 2026?
tags                [Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]
primary_tag                                                                              Politics
winner_token_id    110790003121442365126855864076707686014650523258783405996925622264696084778807
outcome                                                                                       Yes
Name: 59765, dtype: object


In [85]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(dfc) != 0:

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no = pos_per_wallet[pos_per_wallet['outcome'] == 'No'].rename(columns={'position': 'pos_no'})[['dt', 'wallet', 'pos_no']]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt']).reset_index(drop=True)

    # ── 3. weight wallets by proper probabilistic score (Brier skill) with shrinkage
    wallet_scores = (
        train_b_metrics[['wallet', 'brier_skill', 'open_buy_trades']]
        .drop_duplicates('wallet')
        .copy()
    )
    wallet_scores['score'] = wallet_scores['brier_skill'].clip(lower=0.0).fillna(0.0)
    wallet_scores['confidence'] = (
        wallet_scores['open_buy_trades'].fillna(0.0)
        / (wallet_scores['open_buy_trades'].fillna(0.0) + 25.0)
    )
    wallet_scores['wallet_weight'] = wallet_scores['score'] * wallet_scores['confidence']
    wallet_weights = wallet_scores.set_index('wallet')['wallet_weight']

    # ── 4. normalize by each wallet's typical move size in this market
    net_pos['net_step'] = net_pos.groupby('wallet')['net'].diff()
    typical_move = net_pos.groupby('wallet')['net_step'].apply(
        lambda s: s.abs().dropna().median() if s.notna().any() else np.nan
    ).rename('typical_move')
    positive_moves = typical_move[typical_move > 0]
    fallback_move = positive_moves.median() if not positive_moves.empty else 1.0
    typical_move = typical_move.fillna(fallback_move).clip(lower=fallback_move * 0.25)

    # ── 5. carry positions forward so the signal reflects current wallet stance
    timeline = price_ts['dt'].sort_values().drop_duplicates()
    net_panel = (
        net_pos.pivot(index='dt', columns='wallet', values='net')
        .sort_index()
        .reindex(timeline)
        .ffill()
        .fillna(0.0)
    )
    available_wallets = net_panel.columns.intersection(wallet_weights.index).intersection(typical_move.index)
    net_panel = net_panel[available_wallets]
    wallet_weights = wallet_weights.reindex(available_wallets).fillna(0.0)
    typical_move = typical_move.reindex(available_wallets)
    normalized_panel = net_panel.divide(typical_move, axis=1)
    weighted_panel = normalized_panel.multiply(wallet_weights, axis=1)

    signal_ts = pd.DataFrame({
        'dt': net_panel.index,
        'total_net': net_panel.sum(axis=1).to_numpy(),
        'weighted_position': weighted_panel.sum(axis=1).to_numpy(),
    })

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (positive weighted signal => predicts YES)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_panel.abs().max().sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = (
            net_panel[[wallet]].reset_index()
            .rename(columns={wallet: 'net'})
            .sort_values('dt')
        )
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines',
                name=f'{wallet[:8]}...',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['total_net'],
            mode='lines',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── weighted normalized wallet position (primary y)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['weighted_position'],
            mode='lines',
            name='Weighted normalized position',
            line=dict(color='#C44E52', width=3, shape='hv'),
            legendgroup='weighted_signal',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.add_hline(y=0, line=dict(color='#BBBBBB', width=1, dash='dash'))
    fig.update_yaxes(title_text='Net position / weighted normalized total', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions and weighted signal - {MARKET[:16]}...',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [86]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [87]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

71


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
1033,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,Politics,6672445.91,0.79,344675.53
818,0x777fae71d2ff9ec48a1213d48ba1d9d91024a1bb,Finance,2461825.70,0.81,156059.76
459,0x3b4484b6c8cbfdaa383ba337ab3f0d71055e264e,Politics,2064995.62,0.89,143497.64
1379,0xd3989ba133ab48b5b3a81e3dba9b37b5966a46d7,Politics,2242718.97,0.94,98098.82
1422,0xee50a31c3f5a7c77824b12a941a54388a2827ed6,Business,627189.40,0.46,89934.42
671,0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,Politics,1402241.45,0.90,88758.00
541,0x41583f2efc720b8e2682750fffb67f2806fece9f,Politics,2290981.07,0.82,80197.76
207,0x1c144e30f405a25f991cbd8baa15d40599090869,Politics,1188143.90,0.89,77700.91
27,0x03626d34381b0337387b0e8464c898f772009661,Politics,1241574.23,0.90,73226.22
183,0x1955273c5691a1330e264e9daf07411ba913aef1,Politics,1256061.23,0.79,66688.38


In [88]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [89]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [90]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [91]:
import plotly.graph_objects as go

print(f"copyable_group wallets: {len(wallet_cohorts['copyable_group']):,}")
print(f"predicting_group wallets: {len(wallet_cohorts['predicting_group']):,}")

pd.set_option('display.max_rows', 100)

copyable_view = wallet_cohorts['copyable_group'].sort_values('final_score', ascending=False).head(20)
predicting_view = wallet_cohorts['predicting_group'].sort_values('predicting_final_score', ascending=False).head(20)

copyable_view, predicting_view

plot_start = pd.Timestamp(END_DATE_TRAIN, tz='UTC') + pd.Timedelta(days=1)
plot_rows = []
for group_label, cohort_name in [('copyable', 'copyable_group'), ('predicting', 'predicting_group')]:
    wallets = set(wallet_cohorts[cohort_name]['wallet'])
    group_trades = df_full[
        (df_full['wallet'].isin(wallets))
        & (df_full['dt'] >= plot_start)
    ][['dt', 'pnl', 'copyable_pnl']].copy()

    daily = (
        group_trades
        .assign(plot_dt=group_trades['dt'].dt.floor('D'))
        .groupby('plot_dt', as_index=False)[['pnl', 'copyable_pnl']]
        .sum()
        .sort_values('plot_dt')
        .reset_index(drop=True)
    )
    daily['cum_trade_pnl'] = daily['pnl'].cumsum()
    daily['cum_copyable_pnl'] = daily['copyable_pnl'].cumsum()
    daily['wallet_group'] = group_label
    plot_rows.append(daily)

group_perf = pd.concat(plot_rows, ignore_index=True)

fig = go.Figure()
colors = {'copyable': '#1f77b4', 'predicting': '#ff7f0e'}
for group_label in ['copyable', 'predicting']:
    g = group_perf[group_perf['wallet_group'] == group_label]
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_trade_pnl'],
            mode='lines',
            name=f'{group_label} trade pnl',
            line=dict(color=colors[group_label], width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=g['plot_dt'],
            y=g['cum_copyable_pnl'],
            mode='lines',
            name=f'{group_label} copyable pnl',
            line=dict(color=colors[group_label], width=2, dash='dash'),
        )
    )

fig.update_layout(
    title='Stage1 wallet-group performance on test period',
    xaxis_title='Date',
    yaxis_title='Cumulative PnL (USDC)',
    template='plotly_white',
)
fig.show(renderer='browser')


copyable_group wallets: 71
predicting_group wallets: 35


In [92]:
copyable_wallets = wallet_cohorts['copyable_group'].copy().assign(wallet_group='copyable')
predicting_wallets = wallet_cohorts['predicting_group'].copy().assign(wallet_group='predicting')

selected_wallets = (
    pd.concat([copyable_wallets, predicting_wallets], ignore_index=True)
    .drop_duplicates(subset=['wallet', 'wallet_group'])
    .reset_index(drop=True)
)

selected_wallets.to_parquet(WORKSPACE_DIR / 'selected_wallets_v2.parquet', index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")
print(selected_wallets['wallet_group'].value_counts().to_dict())


Saved selected wallets: 106
{'copyable': 71, 'predicting': 35}


In [93]:
WORKSPACE_DIR / "selected_wallets_v2.parquet"


PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [94]:
from polymarket_analysis.strategy.definition import TriggerSpec, WalletSelectionStrategy
from polymarket_analysis.strategy.registry import save_strategy

SIGNAL_THRESHOLD = 0.65

selected_wallets = wallet_cohorts['predicting_group'].copy().reset_index(drop=True)
predicting_wallets = selected_wallets.copy()
if 'wallet_quality' not in predicting_wallets.columns:
    if 'predicting_final_score' in predicting_wallets.columns:
        predicting_wallets['wallet_quality'] = predicting_wallets['predicting_final_score']
    else:
        predicting_wallets['wallet_quality'] = 0.0

strategy_specs = [
    (
        'predicting_group__score_threshold',
        f'predicting_group | score >= {SIGNAL_THRESHOLD:.2f} (Kelly)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.score_threshold',
            params={'threshold': SIGNAL_THRESHOLD, 'dynamic_sizing': True},
            mode='frame',
        ),
    ),
    (
        'predicting_group__all_open_buys',
        'predicting_group | all open-buys (fixed stake)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.all_open_buys',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
    (
        'predicting_group__copy_triggers',
        'predicting_group | copy all triggers (tight slippage)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.copy_triggers',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
]

all_strategies = []
for strategy_id, name, trigger in strategy_specs:
    all_strategies.append(
        WalletSelectionStrategy(
            strategy_id=strategy_id,
            name=name,
            selection_method='predicting_group',
            trigger=trigger,
            wallets=predicting_wallets.copy(),
            params={
                'selection_metric': 'predicting_final_score',
                'top_n': len(predicting_wallets),
                'signal_threshold': SIGNAL_THRESHOLD if trigger.fn_ref.endswith('score_threshold') else None,
            },
            metadata={
                'cohort': 'predicting_group',
                'wallet_group': 'predicting',
                'selection_metric': 'predicting_final_score',
                'signal_threshold': SIGNAL_THRESHOLD,
                'top_n': len(predicting_wallets),
                'end_date_train': str(END_DATE_TRAIN),
                'train_a_end_date': str(TRAIN_A_END_DATE),
                'selection_notes': (
                    'threshold-based: copyable group uses copyability quality gates, '
                    'predicting group uses predictability threshold on remaining wallets'
                ),
            },
        )
    )

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}] wallets={len(strategy.wallets):3d} trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f'\nTotal strategies saved: {len(all_strategies)}')


Saved [predicting_group__score_threshold] wallets= 35 trigger=score_threshold
Saved [predicting_group__all_open_buys] wallets= 35 trigger=all_open_buys
Saved [predicting_group__copy_triggers] wallets= 35 trigger=copy_triggers

Total strategies saved: 3


## Strategy registry summary

In [95]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        'strategy_id': s.strategy_id,
        'name': s.name,
        'selection_method': s.selection_method,
        'num_wallets': len(s.wallets),
        'trigger_fn': s.trigger.fn_ref.split('.')[-1],
        'threshold': s.trigger.params.get('threshold'),
        'dynamic_sizing': s.trigger.params.get('dynamic_sizing'),
    })

pd.DataFrame(summary_rows).sort_values('strategy_id').reset_index(drop=True)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__all_open_buys,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | all open-buys (fixed stake),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,all_open_buys,NaN,False
1,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__copy_triggers,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | copy all triggers (tight slippage),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,copy_triggers,NaN,False
2,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__score_threshold,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | score >= 0.65 (Kelly),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,score_threshold,0.65,True
3,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__all_open_buys,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | all open-buys (fixed stake),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,all_open_buys,NaN,False
4,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__copy_triggers,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | copy all triggers (tight slippage),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,copy_triggers,NaN,False
5,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__score_threshold,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | score >= 0.65 (Kelly),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,score_threshold,0.65,True
6,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__all_open_buys,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | all open-buys (fixed stake),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,all_open_buys,NaN,False
7,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__copy_triggers,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | copy all triggers (tight slippage),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,copy_triggers,NaN,False
8,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__score_threshold,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | score >= 0.65 (Kelly),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,score_threshold,0.65,True
9,0x64e86fefd50cdb6259eb00ff146d9aac03497819__all_open_buys,0x64e86fefd50cdb6259eb00ff146d9aac03497819 | all open-buys (fixed stake),0x64e86fefd50cdb6259eb00ff146d9aac03497819,1,all_open_buys,NaN,False


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [96]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [97]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(33384885, 24366347, 9018538)

In [98]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

8715

In [99]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

8476
3045


,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,last_condition_trade_ts,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x0bfb8009df6c46c1fdd79b65896cf224dc4526a7,0x0e2e5c0146c730a2e4ef8031c9128b4efef451a9f4832f42b46997926f9e1303,2026-06-03 07:28:39+00:00,BUY,200.00,100.00,0.50,50.00,0.00,-50.00,-19.94,False,0.00,2026-06-09 00:24:34+00:00,0x895f8570ab5a5eaac32fa2088892fd4dc7622a5928c7fee6c9336ccd0925fe8b,1,False,39.88,20.34,2.00,2026-06-02T00:00:00Z,Will Karen Bass & Spencer Pratt advance to the second round of the 2026 Los Angeles mayoral elec...,"[Politics, Elections, US Election, Los Angeles, Rewards 300 4.5 50 Deprec, Los Angeles Mayoral E...",Politics,24606039444792628072348728486218619466386821084675596998252795028502077675336,Yes,-50.00,50.00,19.94
1,0x0bfb8009df6c46c1fdd79b65896cf224dc4526a7,0x0e2e5c0146c730a2e4ef8031c9128b4efef451a9f4832f42b46997926f9e1303,2026-06-03 22:22:01+00:00,BUY,300.00,75.00,0.45,33.75,0.00,-33.75,-33.75,False,0.00,2026-06-09 00:24:34+00:00,0x9cd2cda82c998adb30f105fac95d69a4decbe3fc400ba8798ff79f7bce2f273e,1,False,75.00,661.60,12.00,2026-06-02T00:00:00Z,Will Karen Bass & Spencer Pratt advance to the second round of the 2026 Los Angeles mayoral elec...,"[Politics, Elections, US Election, Los Angeles, Rewards 300 4.5 50 Deprec, Los Angeles Mayoral E...",Politics,24606039444792628072348728486218619466386821084675596998252795028502077675336,Yes,-33.75,33.75,33.75


In [100]:
pd.set_option('display.max_colwidth', 100)

In [101]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-06-02 01:40:00+00:00,0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,BUY,0.75,17.94,-0.75,-0.75,3,17.94,1,0.75,-0.75,17.94
1,2026-06-02 02:45:00+00:00,0x12b6a68903cd82ff1b611eacbc09995f6998b6827a77fa89e15e58e96dad3074,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,181.00,200.00,19.00,19.00,1,200.00,1,100.00,10.50,110.50
2,2026-06-02 03:30:00+00:00,0xe0c298bfd778db77c991578b4e7921f7d93098d10575c112af8d905f4893d0e0,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,90.17,102.00,-222.78,-90.17,1,102.00,1,90.17,-90.17,102.00
3,2026-06-02 04:35:00+00:00,0x6fb90f81039cbd7513ab836784737811c8bafdec90c2b0c48be8b907de7f2b06,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,6.51,21.00,-6.51,-6.51,1,21.00,1,6.51,-6.51,21.00
4,2026-06-02 04:40:00+00:00,0x6fb90f81039cbd7513ab836784737811c8bafdec90c2b0c48be8b907de7f2b06,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,1.84,2.59,3.54,0.75,1,2.59,1,1.84,0.75,2.59
5,2026-06-02 06:40:00+00:00,0xd16eec502c46d59efab24b62b5a3123c8850719c18b4b8375bd7a92ed7ab34f2,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,2.40,60.00,-2.40,-2.40,1,60.00,1,2.40,-2.40,60.00
6,2026-06-02 06:55:00+00:00,0x27182c8b13f2a6284754660cbb3c6350a04f7aa4afe04d24dad481947014d90d,0x41583f2efc720b8e2682750fffb67f2806fece9f,BUY,70.01,388.94,-249.58,-70.01,1,388.94,1,70.01,-70.01,388.94
7,2026-06-02 07:15:00+00:00,0x708dd534d004d95ee6de71133e261061fcd254cdbde2faea80691d4d33f70773,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,BUY,9.50,25.00,-9.50,-9.50,1,25.00,1,9.50,-9.50,25.00
8,2026-06-02 07:25:00+00:00,0x71ae0e87ed7f12c78c4677404c5c1df18322223a6f49701d07d85163d804d73e,0x83c56c56bc67c0b99b38abd1c0a3c1aca7a7e193,BUY,23.78,58.00,59.00,34.22,1,58.00,1,23.78,34.22,58.00
9,2026-06-02 08:35:00+00:00,0x0efcc493f7e7578cd5808b29e08c2535dee61c2a1fe841696e76e7892b29cca8,0x92d0cb81e6c891b835c8e0272e8c323095bd269e,BUY,239.70,363.51,-334.25,-239.70,1,363.51,1,100.00,-100.00,151.65


In [102]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [103]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [104]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [105]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e044b076958164f8c5423,146622.48,211661.80,85536.22,46060.66,138,15
0xab1209b73e1ce9ebb4905dad781496385b5102c048889502b174ed733de1506a,4546.91,16505.92,6898.16,5735.50,59,8
0x79c850d5dd022d89aaf4f92105657d7a2e0518d4e4d4cf6cdce44d385fe029a7,4749.19,12556.06,3323.21,3349.69,41,6
0x3094a2b925483a06aa72945a1472e311e5eb6be75284f61e0c008e279508ddf6,7875.75,30963.74,7815.93,3002.90,59,10
0x20af55ab35186377b81219db6cb8615240cba42cea41731091be9484a5f5b122,4989.02,9428.70,6757.90,2908.33,26,7
0x2de8d7b9d103366e37f3a448db8289c5c571dec5c0b9eeddbc673546d4bd9322,3655.90,11037.99,4186.33,2754.54,28,5
0x46dd571d8c32bd58dcc553ae6c2a1ae8dda6d1ec69943bc0f771ea5c35132ca5,5514.33,12292.19,3256.98,2177.81,30,6
0xa48ee32a0cbe5bc1a48844bd14e1691701d3bf8f45f4dd8cb8d1d304561393b6,4562.16,8993.47,3223.34,1893.38,40,8
0xed25d03a2589af03a7a603e7c07c36d53baf38c148e50cc1f5a6a6f285d68862,1278.69,3511.00,8300.82,1752.09,16,5


In [106]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")